In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True  # Proyecto2_RAG: desde la 02, cada etapa lee artefactos de una etapa anterior
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = False

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/facurie/Proyecto2_RAG.git"
    REPO_BRANCH = ""
    REPO_DIR = "Proyecto2_RAG"
    DRIVE = "/content/drive/MyDrive/Proyecto2_RAG"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

# Etapa 0 · Auditoría del corpus: legislación de vivienda y alquiler (BOE)

Fuente: API de **legislación consolidada** del BOE (`https://www.boe.es/datosabiertos/api/legislacion-consolidada`), descargada el **2026-10-24** (la foto original, del 2026-10-16, se perdió: decision_log.md, sección 4). Cada norma trae su texto consolidado con **todas las versiones fechadas** de cada artículo.

Esta notebook responde las ocho preguntas de la auditoría (assignment, Etapa 0) con números, y fija la **regla del subconjunto**:

1. **Universo:** normas consolidadas de 10 materias de vivienda del vocabulario controlado del BOE.
2. **Densidad temática ≥ 0,45:** al menos 45% de los tokens del articulado vigente está en artículos que hablan de vivienda o alquiler. Saca las leyes ómnibus (presupuestos, medidas fiscales, LEC), que entran a la materia porque *un* artículo toca el tema.
3. **`estado_consolidacion = Finalizado`:** saca los textos que el propio BOE marca como desactualizados.

**Cómo correrla.** Localmente, con el `.venv` del repo (`tokenizers`, `lxml`, `pandas`, `requests`). La primera vez baja unas 1.300 respuestas de la API (unos 15 minutos) a `data/raw/boe_cache/`; después lee todo de la caché. No necesita GPU.

**Sale:** `data/subset_v1.json` (las normas del subconjunto con su `fecha_actualizacion`: la versión fijada de la fuente, va a git) y la tabla de auditoría del final, que va a `corpus_card.md` y al informe.

In [ ]:
import json, os, re, sys, warnings, zlib
warnings.filterwarnings("ignore", message="IProgress not found")
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from tokenizers import Tokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    os.environ.setdefault("P2_RAW", "/content/drive/MyDrive/Proyecto2_RAG/raw")  # la foto cruda, en Drive
import boe

# Los tokens se cuentan con el tokenizer del modelo que los lee: el del generador.
TOK = Tokenizer.from_pretrained("Qwen/Qwen3-8B")
def ntok(texts: list[str]) -> list[int]:
    return [len(e.ids) for e in TOK.encode_batch(texts, add_special_tokens=False)] if texts else []

SNAPSHOT = "2026-10-24"
CUTOFF = "20250501"        # Qwen3 se publicó el 2025-04-29: lo vigente desde mayo de 2025 no lo pudo leer
DENSITY_MIN = 0.45
MATERIAS = {"303": "Arrendamientos", "305": "Arrendamientos urbanos", "2476": "Desahucios",
            "5774": "Propiedad Horizontal", "7162": "Viviendas", "7163": "Viviendas de Protección Oficial",
            "7164": "Viviendas de Renta Limitada", "7166": "Viviendas sociales",
            "4118": "Impuesto sobre Viviendas Desocupadas", "1720": "Cooperativas de viviendas"}
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

## 1 · El universo: búsqueda por materia

La API filtra por `materia@codigo` (vocabulario controlado de 6.828 materias). Elegimos las 10 que nombran vivienda, alquiler, desahucio o propiedad horizontal. Se descartaron las de urbanismo, suelo e hipoteca: amplían el corpus hacia temas que nuestras preguntas no van a tocar.

**Control de recall de las materias:** de las normas estatales con "alquiler" en el título, ¿cuántas caen en la unión?

In [ ]:
listing = {}
for code_, name in MATERIAS.items():
    for r in boe.search(f"materia@codigo:{code_}"):
        listing.setdefault(r["identificador"], {"row": r, "materias_query": []})["materias_query"].append(code_)
per_materia = pd.DataFrame([{"materia": f"{c} {n}", "normas": sum(c in v["materias_query"] for v in listing.values())}
                            for c, n in MATERIAS.items()])
print("normas en la unión:", len(listing))
display(per_materia)

alquiler = boe.search("titulo:alquiler and ambito@codigo:1")
fuera = [r for r in alquiler if r["identificador"] not in listing]
print(f"recall sobre títulos con 'alquiler' (estatales): {len(alquiler) - len(fuera)}/{len(alquiler)}")
for r in fuera:
    print("  fuera:", r["identificador"], r["titulo"][:110])

### Un defecto que no tira ningún error: el JSON del análisis pierde materias

El endpoint `/analisis` existe en JSON y en XML. En JSON, la conversión de la API conserva **solo la última** `<materia>` de cada norma. No hay error, no hay advertencia: la LAU queda con una materia, "Viviendas de Protección Oficial", que ni siquiera es su tema. Si hubiéramos clasificado las normas con el JSON, la norma central del corpus habría quedado mal clasificada. Por eso `boe.analisis` lee el XML.

In [ ]:
def n_materias_json(boe_id: str) -> int:
    d = json.loads(boe.get(f"legislacion-consolidada/id/{boe_id}/analisis"))["data"]
    d = d[0] if isinstance(d, list) else d
    return json.dumps(d.get("materias")).count('"codigo"')

cmp = pd.DataFrame([{"id": k, "json": n_materias_json(k), "xml": len(boe.analisis(k)["materias"])} for k in sorted(listing)])
print("LAU:", cmp[cmp.id == "BOE-A-1994-26003"].to_dict("records"))
print(f"normas donde el JSON trae menos materias que el XML: {(cmp.json < cmp.xml).sum()}/{len(cmp)}")
print("materias por norma en JSON:", Counter(cmp.json).most_common(3), "| en XML, mediana:", int(cmp.xml.median()))

## 2 · Descarga y parseo: una pasada por las normas de la unión

Por norma: metadatos (JSON), análisis (XML) y el texto consolidado completo (XML, todas las versiones en una sola llamada). El parser (`boe.parse_texto`) separa en cada versión el **articulado** de las **notas de modificación** (`<blockquote>`, "Se modifica el apartado 3 por..."). Contamos tokens de tres maneras: versión vigente, todas las versiones concatenadas (lo que indexaría un loader ingenuo) y las notas.

In [ ]:
HOUSING = re.compile(r"vivienda|arrendamiento|arrendatari|arrendador|alquiler|desahucio|lanzamiento|inquilin|"
                     r"propiedad horizontal|comunidad de propietarios", re.I)
DEROG_TEXT = re.compile(r"derogad|vigencia agotada|sin efecto|p[ée]rdida de vigencia", re.I)

def whole_norm_derogations(an: dict) -> list[str]:
    # "SE DEROGA" + ", por Ley ..." = la norma entera; "SE DEROGA el art. 5, por ..." = parcial.
    return [p["id_norma"] for p in an["posteriores"]
            if "DEROGA" in (p["relacion"] or "") and re.match(r"^,?\s*por\b", (p["texto"] or "").strip(), re.I)]

DOCS, BLOCKS = [], []
for boe_id in sorted(listing):
    meta, an, blocks = boe.metadatos(boe_id), boe.analisis(boe_id), boe.parse_texto(boe.texto(boe_id))
    cur = ["\n".join(t for _, t in b["versions"][-1]["paras"]) if b["versions"] else "" for b in blocks]
    allv = ["\n".join(t for v in b["versions"] for _, t in v["paras"]) for b in blocks]
    notes = ["\n".join(n for v in b["versions"] for n in v["notes"]) for b in blocks]
    ct, at, nt = ntok(cur), ntok(allv), ntok(notes)
    for b, c, n in zip(blocks, cur, ct):
        last = b["versions"][-1] if b["versions"] else {"paras": [], "n_table": 0, "n_img": 0}
        BLOCKS.append({"doc": boe_id, "block": b["id"], "tipo": b["tipo"], "titulo": b["titulo"],
                       "n_versions": len(b["versions"]), "dates": [v["fecha_vigencia"] or "" for v in b["versions"]],
                       "tokens": n, "housing": bool(HOUSING.search(c)),
                       "first_class": last["paras"][0][0] if last["paras"] else None,
                       "n_table": last["n_table"], "n_img": last["n_img"]})
    nota_inicial = " ".join(c for b, c in zip(blocks, cur) if b["tipo"] == "nota_inicial")
    DOCS.append({"id": boe_id, "titulo": meta["titulo"], "rango": meta["rango"]["texto"], "ambito": meta["ambito"]["texto"],
                 "departamento": meta["departamento"]["texto"], "fecha_disposicion": meta.get("fecha_disposicion"),
                 "fecha_publicacion": meta.get("fecha_publicacion"), "fecha_vigencia": meta.get("fecha_vigencia"),
                 "fecha_actualizacion": meta.get("fecha_actualizacion"), "url_eli": meta.get("url_eli"),
                 "vigencia_agotada": meta.get("vigencia_agotada"), "vigencia_agotada_listado": listing[boe_id]["row"]["vigencia_agotada"],
                 "estatus_derogacion": meta.get("estatus_derogacion"), "fecha_derogacion": meta.get("fecha_derogacion"),
                 "estado_consolidacion": meta["estado_consolidacion"]["texto"], "materias_query": listing[boe_id]["materias_query"],
                 "n_materias": len(an["materias"]), "derog_refs": whole_norm_derogations(an),
                 "nota_inicial": nota_inicial, "texto_dice_derogada": bool(DEROG_TEXT.search(nota_inicial)),
                 "nota_publicacion": next((n for n in an["notas"] if n and n.startswith("Publicada en")), None),
                 "tokens": sum(ct), "tokens_all_versions": sum(at), "tokens_notes": sum(nt)})
docs = pd.DataFrame(DOCS).set_index("id")
blocks = pd.DataFrame(BLOCKS)
prec = blocks[blocks.tipo == "precepto"]
print(f"normas {len(docs)} · bloques {len(blocks):,} · preceptos {len(prec):,}")
print(f"tokens (Qwen3): vigente {docs.tokens.sum():,} · todas las versiones {docs.tokens_all_versions.sum():,} · notas {docs.tokens_notes.sum():,}")

## 3 · La regla del subconjunto: densidad temática

La unión pesa 16 millones de tokens, y sus normas más largas no son de vivienda: entran a una materia de vivienda porque uno de sus artículos la toca. Medimos, por norma, qué fracción de los tokens del articulado vigente está en artículos que mencionan vivienda/alquiler/desahucio/propiedad horizontal.

La distribución es **bimodal**, con un valle entre 0,25 y 0,45: el corte cae en el valle. 0,45 y no 0,5 porque el RDL 11/2020 (COVID: moratoria de desahucios y medidas de alquiler), que queremos adentro, tiene 0,47. Debajo de 0,45 aparecen las leyes de "medidas fiscales y administrativas".

In [ ]:
tok_housing = (prec.tokens * prec.housing).groupby(prec.doc).sum()
docs["density"] = (tok_housing / prec.groupby("doc").tokens.sum()).reindex(docs.index).fillna(0.0)
print("percentiles de densidad:", {p: round(float(np.percentile(docs.density, p)), 2) for p in (10, 25, 50, 75, 90)})
print("histograma (bins de 0,1):", np.histogram(docs.density, bins=10, range=(0, 1))[0].tolist())

rows = []
for th in (0.2, 0.3, 0.4, 0.45, 0.5, 0.6):
    s = docs[docs.density >= th]
    rows.append({"densidad >=": th, "normas": len(s), "tokens": s.tokens.sum(), "estatales": (s.ambito == "Estatal").sum(),
                 "agotadas": (s.vigencia_agotada == "S").sum(), "desactualizadas": (s.estado_consolidacion == "Desactualizado").sum()})
display(pd.DataFrame(rows))

KEY = {"BOE-A-1994-26003": "LAU", "BOE-A-2023-12203": "Ley 12/2023 vivienda", "BOE-A-2019-3108": "RDL 7/2019",
       "BOE-A-1960-10906": "Ley Propiedad Horizontal", "BOE-A-2013-5941": "Ley 4/2013", "BOE-A-2020-4208": "RDL 11/2020 COVID",
       "BOE-A-2018-17293": "RDL 21/2018 (derogado)", "BOE-A-2026-20823": "RDL 29/2026", "BOE-A-2000-323": "LEC",
       "BOE-A-2022-22128": "Presupuestos 2023"}
display(docs.loc[list(KEY), ["density", "tokens", "vigencia_agotada"]].assign(norma=list(KEY.values())))

In [ ]:
excluded_stale = docs[(docs.density >= DENSITY_MIN) & (docs.estado_consolidacion != "Finalizado")]
print(f"excluidas por 'Desactualizado' ({len(excluded_stale)}), todas autonómicas: {set(excluded_stale.ambito)}")
display(excluded_stale[["ambito", "titulo"]])

sub = docs[(docs.density >= DENSITY_MIN) & (docs.estado_consolidacion == "Finalizado")].copy()
print(f"SUBCONJUNTO: {len(sub)} normas, {sub.tokens.sum():,} tokens")
display(pd.crosstab(sub.ambito, sub.vigencia_agotada, margins=True))

out = {"snapshot": SNAPSHOT, "source": "https://www.boe.es/datosabiertos/api/legislacion-consolidada",
       "rule": {"materias": MATERIAS, "density_min": DENSITY_MIN, "density_regex": HOUSING.pattern,
                "estado_consolidacion": "Finalizado"},
       "norms": [{"id": i, "titulo": r.titulo, "ambito": r.ambito, "departamento": r.departamento,
                  "fecha_actualizacion": r.fecha_actualizacion} for i, r in sub.sort_index().iterrows()]}
(ROOT / "data").mkdir(exist_ok=True)
(ROOT / "data" / "subset_v1.json").write_text(json.dumps(out, ensure_ascii=False, indent=1) + "\n", encoding="utf-8", newline="\n")
print("escrito data/subset_v1.json")

## 4 · Tamaño frente a la ventana

¿Entra el subconjunto en la ventana? Con `num_ctx = 8.192` y una reserva de salida de 768 tokens quedan 7.424 para el prompt entero. Y la línea de base honesta "si entra, pegalo": ¿cuántas normas entran **solas** en la ventana?

In [ ]:
PROMPT_BUDGET = 8192 - 768
s_blocks = blocks[blocks.doc.isin(sub.index)]
s_prec = s_blocks[s_blocks.tipo == "precepto"]
print(f"tokens del subconjunto: {sub.tokens.sum():,} = {sub.tokens.sum() / PROMPT_BUDGET:,.0f} ventanas de {PROMPT_BUDGET:,}")
print("tokens por norma:", sub.tokens.describe(percentiles=[.1, .5, .9]).round(0).to_dict())
print(f"normas que entran solas en {PROMPT_BUDGET:,} tokens: {(sub.tokens <= PROMPT_BUDGET).sum()}/{len(sub)}"
      f" · las 5 más largas suman el {100 * sub.tokens.nlargest(5).sum() / sub.tokens.sum():.0f}%")
display(sub.nlargest(8, "tokens")[["tokens", "ambito", "titulo"]])

## 5 · Estructura: ¿por dónde se puede cortar?

Cada norma es una lista de bloques tipados (`precepto`, `encabezado`, `preambulo`, `nota_inicial`, `firma`...), y cada precepto (artículo, disposición) abre con un párrafo `class="articulo"`. El id de bloque es del BOE (`a9`, `a9-2` para el 9 bis, `dt1`...): sirve para el `section_id`, pero no se puede derivar del título.

In [ ]:
print("tipos de bloque:", Counter(s_blocks.tipo).most_common())
print(f"preceptos que abren con class=articulo: {(s_prec.first_class == 'articulo').sum():,}/{len(s_prec):,}")
print("ids de bloque no derivables del título (ejemplos):",
      s_prec[s_prec.block.str.contains("-")][["doc", "block", "titulo"]].head(4).values.tolist())
print("tokens por precepto:", s_prec.tokens.describe(percentiles=[.5, .9]).round(0).to_dict())
print(f"preceptos > 1.500 tokens (tope de padre): {(s_prec.tokens > 1500).sum()} · > 1.000: {(s_prec.tokens > 1000).sum()}")
display(s_prec.nlargest(5, "tokens")[["doc", "block", "titulo", "tokens"]])
print(f"normas con tablas: {(s_blocks.groupby('doc').n_table.sum() > 0).sum()} · con imágenes: "
      f"{(s_blocks.groupby('doc').n_img.sum() > 0).sum()} ({s_blocks.n_img.sum()} imágenes en base64)")

## 6 · Versiones: ¿cambia el corpus?

Cada versión de un bloque trae `fecha_vigencia` y la norma que la introdujo. El Artículo 9 de la LAU (plazo mínimo) muestra por qué importa: el número cambia, vuelve, y vuelve a cambiar, con textos casi idénticos.

In [ ]:
mv = s_prec.n_versions
print(f"preceptos con >1 versión: {(mv > 1).sum():,}/{len(s_prec):,} ({100 * (mv > 1).mean():.0f}%) · con 3+: {(mv >= 3).sum()}")
print("distribución de versiones:", dict(sorted(Counter(mv.clip(upper=6)).items())), "(6 = 6 o más)")
print(f"normas con al menos un precepto multiversión: {s_prec[mv > 1].doc.nunique()}/{len(sub)}")

lau9 = [b for b in boe.parse_texto(boe.texto("BOE-A-1994-26003")) if b["id"] == "a9"][0]
timeline = []
for v in lau9["versions"]:
    t = " ".join(t for _, t in v["paras"])
    m = re.search(r"inferior a (\w+) años(, o inferior a (\w+) años si el arrendador fuese persona jurídica)?", t)
    timeline.append({"vigente desde": v["fecha_vigencia"], "norma": v["id_norma"],
                     "plazo mínimo": (m.group(1) + (f" / {m.group(3)} (persona jurídica)" if m.group(3) else "")) if m else "?",
                     "apartado 4 (vivienda temporal)": "sí" if "vivienda temporal" in t else "no"})
display(pd.DataFrame(timeline))

born = s_prec.dates.map(lambda d: (d[0] if d else "") >= CUTOFF)
modified = s_prec.dates.map(lambda d: any(x >= CUTOFF for x in d[1:]))
print(f"preceptos que nacieron después del corte de Qwen3 ({CUTOFF}): {born.sum()} ({100 * born.mean():.0f}%)"
      f" · modificados después: {modified.sum()} ({100 * modified.mean():.0f}%)")
print(f"normas publicadas después del corte: {(sub.fecha_publicacion >= CUTOFF).sum()} "
      f"({sub[sub.fecha_publicacion >= CUTOFF].tokens.sum():,} tokens)")
print(f"loader ingenuo: todas las versiones = {sub.tokens_all_versions.sum():,} tokens; notas = {sub.tokens_notes.sum():,};"
      f" vigente = {sub.tokens.sum():,}")

## 7 · Metadatos: cobertura y ¿mienten?

Cobertura: qué fracción tiene un valor utilizable. Confiabilidad: contrastamos `vigencia_agotada` con otras tres señales (el campo `estatus_derogacion`, las referencias "SE DEROGA" de la norma entera, y la nota inicial del propio texto).

In [ ]:
cov = {f: f"{sub[f].notna().sum()}/{len(sub)}" for f in ["fecha_disposicion", "fecha_publicacion", "fecha_vigencia", "url_eli", "fecha_derogacion"]}
cov["materias >= 1"] = f"{(sub.n_materias > 0).sum()}/{len(sub)}"
print("cobertura:", cov)
print("listado vs metadatos (misma foto):", Counter(zip(sub.vigencia_agotada_listado, sub.vigencia_agotada)))
display(pd.crosstab(sub.vigencia_agotada, sub.estatus_derogacion, rownames=["vigencia_agotada"], colnames=["estatus_derogacion"]))
display(pd.crosstab(sub.vigencia_agotada, sub.texto_dice_derogada, rownames=["vigencia_agotada"], colnames=["nota inicial dice derogada"]))
display(pd.crosstab(sub.vigencia_agotada, sub.derog_refs.map(bool), rownames=["vigencia_agotada"], colnames=["ref. SE DEROGA (norma entera)"]))
disagree = sub[(sub.vigencia_agotada == "S") != sub.texto_dice_derogada]
display(disagree[["vigencia_agotada", "estatus_derogacion", "fecha_derogacion", "nota_inicial"]].assign(
    nota_inicial=disagree.nota_inicial.str[:150]))

Tres trampas más que no tiran error:

- **Fechas invertidas.** En las autonómicas, `fecha_publicacion` es la del BOE, que llega después que la del boletín regional (la fecha regional está en una nota del análisis). Para preguntas temporales hay que usar `fecha_vigencia`.
- **La misma región con dos nombres** de departamento: un filtro por región tomado de la memoria del usuario se perdería normas.
- **Normas periódicas que se reemplazan sin derogarse:** las resoluciones anuales de tipos de interés de los préstamos convenidos figuran todas como vigentes.

In [ ]:
inv = sub[sub.fecha_vigencia.notna() & (sub.fecha_vigencia < sub.fecha_publicacion)]
print(f"fecha_vigencia < fecha_publicacion: {len(inv)}/{len(sub)} · por ámbito: {dict(Counter(inv.ambito))}")
ex = inv[inv.ambito == "Autonómico"].iloc[0]
print("  ejemplo:", ex.name, ex.titulo[:60], ex[["fecha_publicacion", "fecha_vigencia", "nota_publicacion"]].to_dict())
print("  la estatal:", inv[inv.ambito == "Estatal"].titulo.str[:70].tolist(), "(entra en vigor 'simultáneamente con la Ley del ITP'; el BOE registra el 1/7/1980 que fija una de sus disposiciones: un valor derivado)")
auto = sub[sub.ambito == "Autonómico"]
print("departamentos autonómicos con 'Valencia':", Counter(d for d in auto.departamento if "Valencia" in d))
per = sub[sub.titulo.str.contains("tipos de interés efectivos anuales")]
print(f"resoluciones anuales de tipos de interés: {len(per)}, años {sorted(per.fecha_disposicion.str[:4])},"
      f" con vigencia_agotada=N: {(per.vigencia_agotada == 'N').sum()}")

## 8 · Casi-duplicados: ¿cuán parecidos son los documentos entre sí?

Tres niveles, cada uno lo paga una etapa distinta:

- **norma ↔ norma** (Etapa 2, gold como conjunto o como versiones): MinHash sobre shingles de 5 palabras para todos los pares, Jaccard exacto y contención sobre los candidatos;
- **artículo ↔ artículo de otra norma** (Etapa 3, vecinos confundibles): LSH con 32 bandas de 4 filas, Jaccard exacto ≥ 0,7;
- **versión ↔ versión del mismo artículo** (filtros por fecha y preguntas temporales).

Medimos texto copiado, no "temas parecidos": eso lo hacen los embeddings, y lo veremos en la Etapa 3.

In [ ]:
N_PERM, P = 128, 4294967311
rng = np.random.default_rng(0)
A, B = rng.integers(1, 2**31, N_PERM, dtype=np.uint64), rng.integers(0, 2**32, N_PERM, dtype=np.uint64)

def shingles(text: str, k: int = 5) -> set[int]:
    w = re.findall(r"\w+", text.lower())
    return {zlib.crc32(" ".join(w[i:i + k]).encode()) for i in range(max(len(w) - k + 1, 0))}

def minhash(sh: set[int]) -> np.ndarray:
    if not sh:
        return np.full(N_PERM, P, dtype=np.uint64)
    x = np.fromiter(sh, dtype=np.uint64)
    return ((A[:, None] * x[None, :] + B[:, None]) % P).min(axis=1)

jacc = lambda a, b: len(a & b) / max(len(a | b), 1)
ids = sorted(sub.index)
doc_sh, art, vsims = {}, [], []
for boe_id in ids:
    cur_all = []
    for b in boe.parse_texto(boe.texto(boe_id)):
        if not b["versions"]:
            continue
        texts = ["\n".join(t for _, t in v["paras"]) for v in b["versions"]]
        cur_all.append(texts[-1])
        if b["tipo"] == "precepto":
            s = shingles(texts[-1])
            if len(s) >= 40:
                art.append((boe_id, b["id"], s))
            vsims += [jacc(shingles(x), shingles(y)) for x, y in zip(texts, texts[1:]) if x and y]
    doc_sh[boe_id] = shingles("\n".join(cur_all))

sig = np.stack([minhash(doc_sh[i]) for i in ids])
est = (sig[:, None, :] == sig[None, :, :]).mean(axis=2)
pairs = []
for i, j in zip(*np.where(np.triu(est, 1) >= 0.3)):
    a, b = doc_sh[ids[i]], doc_sh[ids[j]]
    pairs.append({"a": ids[i], "b": ids[j], "jaccard": round(jacc(a, b), 2),
                  "contención": round(len(a & b) / max(min(len(a), len(b)), 1), 2),
                  "título a": sub.titulo[ids[i]][:60], "título b": sub.titulo[ids[j]][:60]})
pairs = pd.DataFrame(pairs).sort_values("contención", ascending=False)
print(f"pares de normas con Jaccard estimado >= 0,3: {len(pairs)}")
display(pairs.head(15))

In [ ]:
asig = np.stack([minhash(s) for _, _, s in art])
buckets = {}
for idx in range(len(art)):
    for band in range(32):
        buckets.setdefault((band, asig[idx, band * 4:(band + 1) * 4].tobytes()), []).append(idx)
cand = {p for m in buckets.values() if 1 < len(m) < 200 for p in combinations(m, 2)}
twins = [(art[i][0], art[i][1], art[j][0], art[j][1]) for i, j in cand
         if art[i][0] != art[j][0] and jacc(art[i][2], art[j][2]) >= 0.7]
with_twin = {(a, b) for a, b, _, _ in twins} | {(c, d) for _, _, c, d in twins}
print(f"artículos (>= 40 shingles) con un gemelo casi literal en OTRA norma: {len(with_twin)}/{len(art)} ({100 * len(with_twin) / len(art):.0f}%)")
print("pares de normas que más artículos comparten:")
for (a, c), n in Counter(tuple(sorted((a, c))) for a, _, c, _ in twins).most_common(6):
    print(f"  {n:>3}  {sub.titulo[a][:55]}  <->  {sub.titulo[c][:55]}")
vs = np.array(vsims)
print(f"\nversiones consecutivas del mismo artículo: {vs.size} pares · Jaccard mediana {np.median(vs):.2f} · "
      f"p10 {np.percentile(vs, 10):.2f} · {100 * (vs >= 0.8).mean():.0f}% con Jaccard >= 0,8")

## 9 · La tabla de la auditoría

Números del subconjunto (177 normas) salvo que se diga otra cosa. La última columna es lo que importa: **qué etapa paga cada "no"**.

| # | Propiedad | Medido | ¿Cumple? | Qué etapa paga |
|---|---|---|---|---|
| 1 | ¿Está la respuesta, y en qué idioma? | Español jurídico peninsular. Temas: duración y prórroga, renta y actualización, fianza, desahucio, zonas tensionadas, VPO, propiedad horizontal. 92 estatales + 85 autonómicas. Faltan: el procedimiento de desahucio de la LEC y la ley catalana de vivienda (excluidas) | sí, con huecos conocidos | Etapa 2: los huecos son las preguntas `out_of_corpus` adyacentes |
| 2 | ¿Texto limpio? ¿Qué deja pasar un loader ingenuo? | Concatenar versiones lleva el subconjunto de 3,09 M a 4,46 M tokens; las notas de modificación suman 0,67 M más. 23% de las normas con tablas, 15% con imágenes en base64. "m2" y "m²" conviven. El JSON del análisis pierde materias | no | Etapa 1: versión vigente + notas aparte; tablas; imágenes fuera; normalización (con cuidado: NFKC cambia "1.º" por "1.o") |
| 3 | ¿Estructura para cortar? | Bloques tipados; 99% de los preceptos abren con `class="articulo"`. Mediana de 230 tokens por precepto; 3% supera 1.500 (hasta 25 K en "Artículo único" modificadores) | sí | Etapa 3: cortar los preceptos gigantes por apartados; padre = precepto con tope |
| 4 | ¿Id estable? | `BOE-A-AAAA-NNNNN` oficial; id de bloque del BOE (`a9`, `a9-2`). Un mismo id de bloque puede cambiar por completo de contenido entre versiones (arts. 38-40 de la LAU) | sí, con la versión fijada | Etapa 1: `section_id = doc_id#bloque`; Etapa 2: el gold apunta a la foto del 2026-10-16 |
| 5 | ¿Metadatos correctos? | Cobertura: `fecha_vigencia` 98%, ELI 94%, materias 100%. `vigencia_agotada` coincide con `estatus_derogacion` en 176/177. 1 norma "vigente" que su texto dice derogada (Ley balear 11/2005). 65 (37%) con `fecha_publicacion` posterior a la entrada en vigor (las autonómicas). Resoluciones anuales reemplazadas, todas "vigentes". La misma región con dos nombres | casi | Etapa 1: mapeo de estado con correcciones; fechas desde `fecha_vigencia`; región normalizada. Etapa 6: el filtro por región o fecha hereda estos errores |
| 6 | ¿Cambia? ¿Cómo te enterarías? | 18% de los preceptos con más de una versión (8% con 3+). La LAU cambió el 1, el 2 y el 8 de octubre de 2026. El listado acepta `from=AAAAMMDD` para ver lo actualizado desde una fecha | sí, mucho | Etapa 1: versión fijada (`data/subset_v1.json` con `fecha_actualizacion`). Etapas 2 y 6: preguntas "a una fecha" y actualizaciones de memoria |
| 7 | ¿Se puede armar un eval set? | Sí: identificadores literales (números de ley, artículos), fechas de vigencia, versiones con respuestas distintas, normas de 2026 que el generador no pudo leer (11% de los preceptos nació después de mayo de 2025) | sí | Etapa 2 |
| 8 | ¿Cuán parecidos son entre sí? | 7% de los artículos tiene un gemelo casi literal en otra norma. Pares RDL derogado ↔ RDL re-decretado (26 ↔ 29/2026, 21/2018 ↔ 7/2019); decreto-ley ↔ ley; 12 resoluciones anuales con plantilla. Versiones consecutivas: Jaccard mediana 0,82 | muy parecidos | Etapa 1: versiones, no copias; Etapa 2: gold con fecha o conjunto; Etapa 3: prefijo con año y estado |

## 10 · Diez documentos leídos de punta a punta

La auditoría pide diez documentos crudos leídos por una persona. Los elegimos para cubrir los casos que los números marcan.

| Norma | Por qué leerla |
|---|---|
| BOE-A-1994-26003 · LAU | el centro del corpus; 9 versiones del art. 9 |
| BOE-A-2023-12203 · Ley 12/2023 | definiciones (gran tenedor, zona tensionada) modificadas en 2026 |
| BOE-A-2026-20823 · RDL 29/2026 | el re-decreto de octubre de 2026; posterior al corte |
| BOE-A-2018-17293 · RDL 21/2018 | derogado al mes; sus versiones siguen en la LAU |
| BOE-A-1960-10906 · Ley de Propiedad Horizontal | ley vieja, muy modificada |
| BOE-A-2020-4208 · RDL 11/2020 | ómnibus COVID que entra por densidad (0,47) |
| BOE-A-2005-21319 · Ley balear 11/2005 | el metadato que miente |
| BOE-A-2024-7593 · Resolución tipos de interés 2024 | plantilla anual, tablas |
| BOE-A-2018-9774 · Ley 5/2018 de vivienda de Illes Balears | fecha del BOE posterior a la entrada en vigor: la del boletín regional está en una nota |
| BOE-A-2015-10727 · el "Artículo único" de 25 K tokens | qué significa cortar un precepto gigante |